# 캠퍼스 장애물 인식: LoveDA로 분할 모델 학습 후 영남대 정사영상 추론

- 모델: torchvision DeepLabV3 + ResNet-50 (ImageNet 사전 학습 인코더), PyTorch
- 학습 데이터: LoveDA (0.3 m 항공영상, 7개 클래스), Zenodo에서 내려받음 → **Settings에서 Internet On 필요**
- 추론 데이터: 비공개 Kaggle 데이터셋으로 올린 `campus_25cm.png` (Add Input으로 연결)
- 출력(`/kaggle/working`): `best.pth`, `train_log.json`, `campus_pred_0.3m.png`, `campus_pred_preview.jpg`

실행: Accelerator를 GPU(T4 또는 P100)로 두고 **Save Version → Save & Run All**. 브라우저를 닫아도 끝까지 돌아갑니다.

In [ ]:
import glob, json, math, os, random, subprocess, time, zipfile
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet50_Weights
from torchvision.models.segmentation import deeplabv3_resnet50

Image.MAX_IMAGE_PIXELS = None
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print(torch.__version__, DEV, torch.cuda.get_device_name(0) if DEV == 'cuda' else '')

EPOCHS = 30            # 약 1~1.5시간 (T4 기준)
CROP = 512
BATCH = 8
LR = 2e-4
TIME_LIMIT_H = 8.0     # 이 시간을 넘기면 학습을 멈추고 추론으로 넘어감
OUT = '/kaggle/working'
DATA = '/tmp/loveda'

# LoveDA 마스크 값: 0 no-data, 1 background, 2 building, 3 road, 4 water, 5 barren, 6 forest, 7 agriculture
CLASSES = ['background', 'building', 'road', 'water', 'barren', 'forest', 'agriculture']
NC = len(CLASSES)
MEAN = np.array([0.485, 0.456, 0.406], np.float32) * 255
STD = np.array([0.229, 0.224, 0.225], np.float32) * 255

## 1. LoveDA 내려받기 (Zenodo, Train ≈ 4 GB / Val ≈ 2.4 GB)

In [ ]:
os.makedirs(DATA, exist_ok=True)
for split in ('Train', 'Val'):
    if os.path.isdir(f'{DATA}/{split}'):
        continue
    url = f'https://zenodo.org/records/5706578/files/{split}.zip?download=1'
    zpath = f'/tmp/{split}.zip'
    subprocess.run(['wget', '-q', '-O', zpath, url], check=True)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(DATA)
    os.remove(zpath)
for split in ('Train', 'Val'):
    for scene in ('Urban', 'Rural'):
        print(split, scene, len(glob.glob(f'{DATA}/{split}/{scene}/images_png/*.png')))

In [ ]:
def pairs(split):
    imgs = sorted(glob.glob(f'{DATA}/{split}/*/images_png/*.png'))
    return [(p, p.replace('images_png', 'masks_png')) for p in imgs]


def to_tensor(img):
    return torch.from_numpy(((img.astype(np.float32) - MEAN) / STD).transpose(2, 0, 1))


def to_label(mask):
    lab = mask.astype(np.int64) - 1
    lab[mask == 0] = 255          # no-data는 학습에서 무시
    return torch.from_numpy(lab)


class LoveDA(Dataset):
    """학습: 무작위 축척(0.25 m 캠퍼스와 0.3 m LoveDA 차이 포함)·자르기·뒤집기·회전·밝기 변화."""

    def __init__(self, items, train):
        self.items, self.train = items, train

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        ip, mp = self.items[i]
        img, mask = Image.open(ip).convert('RGB'), Image.open(mp)
        if not self.train:
            return to_tensor(np.asarray(img)), to_label(np.asarray(mask))
        s = random.uniform(0.75, 1.35)
        size = int(1024 * s)
        img = img.resize((size, size), Image.BILINEAR)
        mask = mask.resize((size, size), Image.NEAREST)
        x, y = random.randint(0, size - CROP), random.randint(0, size - CROP)
        img = np.asarray(img)[y:y + CROP, x:x + CROP]
        mask = np.asarray(mask)[y:y + CROP, x:x + CROP]
        if random.random() < 0.5:
            img, mask = img[:, ::-1], mask[:, ::-1]
        k = random.randint(0, 3)
        img, mask = np.rot90(img, k), np.rot90(mask, k)
        img = img.astype(np.float32) * random.uniform(0.8, 1.2) + random.uniform(-20, 20)
        img = np.clip(img, 0, 255)
        return to_tensor(np.ascontiguousarray(img)), to_label(np.ascontiguousarray(mask))


train_items, val_items = pairs('Train'), pairs('Val')
train_dl = DataLoader(LoveDA(train_items, True), BATCH, shuffle=True, num_workers=4, drop_last=True,
                      pin_memory=True, persistent_workers=True)
val_dl = DataLoader(LoveDA(val_items, False), 4, num_workers=4, pin_memory=True)
print(len(train_items), 'train /', len(val_items), 'val images')

## 2. 모델과 학습

In [ ]:
model = deeplabv3_resnet50(weights=None, weights_backbone=ResNet50_Weights.IMAGENET1K_V2,
                           num_classes=NC, aux_loss=True).to(DEV)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
steps = EPOCHS * len(train_dl)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.05)
scaler = torch.amp.GradScaler()
crit = nn.CrossEntropyLoss(ignore_index=255)


@torch.no_grad()
def evaluate():
    model.eval()
    inter, union = torch.zeros(NC), torch.zeros(NC)
    for x, y in val_dl:
        x, y = x.to(DEV), y.to(DEV)
        with torch.autocast('cuda', dtype=torch.float16):
            p = model(x)['out'].argmax(1)
        valid = y != 255
        for c in range(NC):
            pc, yc = (p == c) & valid, (y == c) & valid
            inter[c] += (pc & yc).sum().item()
            union[c] += (pc | yc).sum().item()
    iou = (inter / union.clamp(min=1)).tolist()
    model.train()
    return dict(zip(CLASSES, [round(v, 4) for v in iou])), round(float(np.mean(iou)), 4)


log, best, t0 = [], -1.0, time.time()
model.train()
for ep in range(1, EPOCHS + 1):
    total = 0.0
    for x, y in train_dl:
        x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.float16):
            out = model(x)
            loss = crit(out['out'], y) + 0.4 * crit(out['aux'], y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        total += loss.item()
    entry = {'epoch': ep, 'loss': round(total / len(train_dl), 4), 'min': round((time.time() - t0) / 60, 1)}
    last = ep == EPOCHS or (time.time() - t0) / 3600 > TIME_LIMIT_H
    if ep % 5 == 0 or last:
        entry['val_iou'], entry['val_miou'] = evaluate()
        if entry['val_miou'] > best:
            best = entry['val_miou']
            torch.save(model.state_dict(), f'{OUT}/best.pth')
    log.append(entry)
    print(entry, flush=True)
    with open(f'{OUT}/train_log.json', 'w') as f:
        json.dump(log, f, indent=1)
    if last:
        break
print('best val mIoU', best)

## 3. 캠퍼스 정사영상 추론
0.25 m 영상을 LoveDA와 같은 0.3 m로 줄인 뒤, 1024 px 창을 768 px 간격으로 밀며 추론하고 겹친 곳은 확률을 평균합니다.

In [ ]:
found = glob.glob('/kaggle/input/**/campus_25cm.png', recursive=True)
assert found, 'campus_25cm.png 데이터셋을 Add Input으로 연결하세요'
src = Image.open(found[0]).convert('RGB')
w, h = src.size
img = np.asarray(src.resize((round(w * 0.25 / 0.3), round(h * 0.25 / 0.3)), Image.BILINEAR))
H, W = img.shape[:2]
print('campus', (w, h), '->', (W, H), 'at 0.3 m')

model.load_state_dict(torch.load(f'{OUT}/best.pth', map_location=DEV))
model.eval()
WIN, STRIDE = 1024, 768
prob = torch.zeros(NC, H, W)
count = torch.zeros(1, H, W)
ys = list(range(0, max(H - WIN, 0) + 1, STRIDE)) + [max(H - WIN, 0)]
xs = list(range(0, max(W - WIN, 0) + 1, STRIDE)) + [max(W - WIN, 0)]
with torch.no_grad():
    for y0 in sorted(set(ys)):
        for x0 in sorted(set(xs)):
            tile = to_tensor(img[y0:y0 + WIN, x0:x0 + WIN])[None].to(DEV)
            with torch.autocast('cuda', dtype=torch.float16):
                p = model(tile)['out'].softmax(1) + model(tile.flip(3))['out'].softmax(1).flip(3)
            th, tw = tile.shape[2:]
            prob[:, y0:y0 + th, x0:x0 + tw] += p[0].float().cpu()
            count[:, y0:y0 + th, x0:x0 + tw] += 2
pred = (prob / count).argmax(0).numpy().astype(np.uint8) + 1   # LoveDA 값(1~7)으로 저장
Image.fromarray(pred).save(f'{OUT}/campus_pred_0.3m.png')

PALETTE = np.array([[0, 0, 0], [255, 255, 255], [255, 0, 0], [255, 255, 0], [0, 0, 255],
                    [159, 129, 183], [0, 255, 0], [255, 195, 128]], np.uint8)
blend = (0.5 * img + 0.5 * PALETTE[pred]).astype(np.uint8)
Image.fromarray(blend).reduce(3).save(f'{OUT}/campus_pred_preview.jpg', quality=85)
share = {c: round(float((pred == i + 1).mean()), 3) for i, c in enumerate(CLASSES)}
print('class share on campus:', share)